In [5]:
#algoritmo de score
import pandas as pd
import numpy as np

products = r"CAMINHO DO SEU ARQUIVO"
products_missing = r"CAMINHO DO SEU ARQUIVO"
data_customers = r"CAMINHO DO SEU ARQUIVO"
data_drivers = r"CAMINHO DO SEU ARQUIVO"
orders = r"CAMINHO DO SEU ARQUIVO"

df_orders = pd.read_csv(orders)
df_missing = pd.read_csv(products_missing)
df_drivers = pd.read_csv(data_drivers)
df_customers = pd.read_csv(data_customers)
df_products = pd.read_csv(products).rename(columns={'produc_id': 'product_id'})
df_products['price_clean'] = df_products['price'].str.replace('$', '', regex=False).astype(float)

df_missing_long = df_missing.melt(
    id_vars=['order_id'], 
    value_vars=['product_id_1', 'product_id_2', 'product_id_3'], 
    value_name='product_id'
).dropna()
df_missing_long = df_missing_long[df_missing_long['product_id'].str.strip() != '']
df_detalhes_problemas = df_missing_long.merge(df_products, on='product_id', how='inner')
df_resumo_pedidos = df_detalhes_problemas.groupby('order_id').agg(
    valor_perdido=('price_clean', 'sum'),
    qtd_eletronicos=('category', lambda x: (x == 'Eletrônicos').sum())
).reset_index()
df_base_pedidos = df_orders.merge(df_resumo_pedidos, on='order_id', how='left')
df_base_pedidos['valor_perdido'] = df_base_pedidos['valor_perdido'].fillna(0)
df_base_pedidos['qtd_eletronicos'] = df_base_pedidos['qtd_eletronicos'].fillna(0)
df_base_pedidos['teve_problema'] = np.where(df_base_pedidos['valor_perdido'] > 0, 1, 0)
perfil_drivers = df_base_pedidos.groupby('driver_id').agg(
    total_pedidos=('order_id', 'count'),
    pedidos_com_problema=('teve_problema', 'sum'),
    prejuizo_total=('valor_perdido', 'sum'),
    total_eletronicos_perdidos=('qtd_eletronicos', 'sum')
).reset_index()
perfil_drivers['taxa_problema'] = (perfil_drivers['pedidos_com_problema'] / perfil_drivers['total_pedidos']) * 100
perfil_drivers['score_fraude'] = (
    (perfil_drivers['taxa_problema'] > 15).astype(int) +
    (perfil_drivers['prejuizo_total'] > 100).astype(int) +
    (perfil_drivers['total_eletronicos_perdidos'] > 0).astype(int) * 2
)
df_final_drivers = perfil_drivers.merge(df_drivers, on='driver_id', how='inner')
df_final_drivers = df_final_drivers.rename(columns={'driver_id': 'id_unico', 'driver_name': 'nome'})
df_final_drivers['tipo_perfil'] = 'Entregador'
perfil_customers = df_base_pedidos.groupby('customer_id').agg(
    total_pedidos=('order_id', 'count'),
    pedidos_com_problema=('teve_problema', 'sum'),
    prejuizo_total=('valor_perdido', 'sum'),
    total_eletronicos_perdidos=('qtd_eletronicos', 'sum')
).reset_index()

perfil_customers['taxa_problema'] = (perfil_customers['pedidos_com_problema'] / perfil_customers['total_pedidos']) * 100

perfil_customers['score_fraude'] = (
    (perfil_customers['taxa_problema'] > 15).astype(int) +
    (perfil_customers['prejuizo_total'] > 100).astype(int) +
    (perfil_customers['total_eletronicos_perdidos'] > 0).astype(int) * 2
)

df_final_customers = perfil_customers.merge(df_customers, on='customer_id', how='inner')
df_final_customers = df_final_customers.rename(columns={'customer_id': 'id_unico', 'customer_name': 'nome'})
df_final_customers['tipo_perfil'] = 'Cliente'
df_tabela_mestre = pd.concat([df_final_drivers, df_final_customers], ignore_index=True)
df_tabela_mestre = df_tabela_mestre[[
    'id_unico', 'nome', 'tipo_perfil', 'total_pedidos', 
    'pedidos_com_problema', 'taxa_problema', 'prejuizo_total', 
    'total_eletronicos_perdidos', 'score_fraude'
]]
def interpretar_score(score):
    if score <= 1:
        return 'Erro operacional comum'
    elif score <= 3:
        return 'Comportamento suspeito (Requer auditoria leve)'
    else:
        return 'Altíssima probabilidade de fraude (Foco da sua análise final!)'

df_tabela_mestre['interpretacao'] = df_tabela_mestre['score_fraude'].apply(interpretar_score)
df_tabela_mestre.to_json('~/Downloads/tabela_score.json', orient='index', indent=4)
df_tabela_mestre.to_csv('~/Downloads/tabela_score.csv', index=False)

print("--- NOVO PROCESSO EXECUTADO COM SUCESSO ---")

--- NOVO PROCESSO EXECUTADO COM SUCESSO ---
Arquivo JSON estruturado para abertura de colunas automática gerado em Downloads.
